In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Model Visualization Demo\n",
    "\n",
    "This notebook visualizes the trained model's performance."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('..')\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import torch\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from torch.utils.data import DataLoader\n",
    "\n",
    "from config import cfg\n",
    "from data.dataset import PhotometryDataset\n",
    "from models.transformer import PhotometryTransformer\n",
    "from models.masking import apply_mask_fixed\n",
    "from training.trainer import reconstruction_loss\n",
    "from evaluate import evaluate_model, compute_statistics\n",
    "from visualize import (\n",
    "    plot_training_curves,\n",
    "    plot_reconstruction_scatter,\n",
    "    plot_residual_histograms,\n",
    "    plot_tsne,\n",
    "    plot_masking_example\n",
    ")\n",
    "from utils.helpers import set_global_seed, setup_logging\n",
    "\n",
    "sns.set_theme(style=\"whitegrid\")\n",
    "set_global_seed(cfg.seed)\n",
    "logger = setup_logging()\n",
    "device = torch.device(cfg.device)\n",
    "\n",
    "print(f\"Using device: {device}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load validation dataset\n",
    "data = np.load(cfg.output_npz, allow_pickle=True)\n",
    "\n",
    "val_dataset = PhotometryDataset(\n",
    "    data['val_values'],\n",
    "    data['val_mask'],\n",
    "    data['val_band'],\n",
    "    data['val_survey'],\n",
    "    data['val_type'],\n",
    "    np.zeros((len(data['val_values']), cfg.global_features))\n",
    ")\n",
    "\n",
    "val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)\n",
    "print(f\"Validation samples: {len(val_dataset)}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load model\n",
    "band_vocab = dict(data['band_vocab'].item())\n",
    "survey_vocab = dict(data['survey_vocab'].item())\n",
    "type_vocab = dict(data['type_vocab'].item())\n",
    "\n",
    "model = PhotometryTransformer(\n",
    "    cfg, \n",
    "    len(band_vocab), \n",
    "    len(survey_vocab), \n",
    "    len(type_vocab)\n",
    ").to(device)\n",
    "\n",
    "# Load best model\n",
    "checkpoint = torch.load(cfg.best_model_path, map_location=device, weights_only=False)\n",
    "model.load_state_dict(checkpoint['model_state_dict'])\n",
    "print(f\"Loaded best model from epoch {checkpoint['epoch']} with val_loss: {checkpoint['best_val_loss']:.6f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Evaluate model\n",
    "all_real, all_pred, metrics = evaluate_model(model, val_loader, device, cfg.num_mask)\n",
    "\n",
    "print(\"=\"*50)\n",
    "print(\"RECONSTRUCTION METRICS\")\n",
    "print(\"=\"*50)\n",
    "for name in ['Flux', 'Error']:\n",
    "    print(f\"\\n{name}:\")\n",
    "    for metric, value in metrics[name].items():\n",
    "        print(f\"  {metric}: {value:.4f}\")\n",
    "print(\"=\"*50)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize reconstruction scatter\n",
    "plot_reconstruction_scatter(\n",
    "    all_real, all_pred,\n",
    "    metrics['Flux']['RMSE'],\n",
    "    metrics['Error']['RMSE'],\n",
    "    save_path='reconstruction_scatter.pdf'\n",
    ")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize residuals\n",
    "plot_residual_histograms(all_real, all_pred, save_path='residual_histograms.pdf')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Compute detailed statistics\n",
    "stats = compute_statistics(all_real, all_pred)\n",
    "\n",
    "print(\"=\"*50)\n",
    "print(\"RECONSTRUCTION STATISTICS\")\n",
    "print(\"=\"*50)\n",
    "for name in ['Flux', 'Error']:\n",
    "    print(f\"\\n{name}:\")\n",
    "    for stat, value in stats[name].items():\n",
    "        print(f\"  {stat}: {value}\")\n",
    "print(\"=\"*50)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize t-SNE\n",
    "plot_tsne(model, val_loader, device, save_path='tsne_visualization.pdf')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize masking example\n",
    "sample_idx = 0\n",
    "valid = val_dataset.mask[sample_idx].numpy().astype(bool)\n",
    "mask_positions = apply_mask_fixed(\n",
    "    val_dataset.mask[sample_idx].unsqueeze(0).to(device),\n",
    "    num_mask=cfg.num_mask\n",
    ")[0].cpu().numpy()\n",
    "\n",
    "plot_masking_example(valid, mask_positions, save_path='masking_example.pdf')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Sample reconstruction visualization\n",
    "model.eval()\n",
    "with torch.no_grad():\n",
    "    batch = next(iter(val_loader))\n",
    "    values = batch[\"values\"].to(device)\n",
    "    band = batch[\"band_ids\"].to(device)\n",
    "    survey = batch[\"survey_ids\"].to(device)\n",
    "    type_id = batch[\"type_ids\"].to(device)\n",
    "    valid_mask = batch[\"valid_mask\"].to(device)\n",
    "    global_feat = batch[\"global_features\"].to(device)\n",
    "\n",
    "    pred, _, mask = model(values, band, survey, type_id, global_feat, valid_mask)\n",
    "\n",
    "# Plot for first sample\n",
    "idx = 0\n",
    "mask_pos = mask[idx].cpu().numpy().astype(bool)\n",
    "\n",
    "fig, axes = plt.subplots(2, 1, figsize=(14, 6))\n",
    "\n",
    "# Flux\n",
    "axes[0].plot(values[idx, :, 0].cpu().numpy(), 'b-', label='True', alpha=0.7)\n",
    "axes[0].plot(pred[idx, :, 0].cpu().numpy(), 'r--', label='Predicted', alpha=0.7)\n",
    "axes[0].scatter(np.where(mask_pos)[0], values[idx, mask_pos, 0].cpu().numpy(), \n",
    "               color='green', s=50, marker='o', label='Masked', zorder=5)\n",
    "axes[0].set_xlabel('Measurement Index')\n",
    "axes[0].set_ylabel('Flux (arcsinh)')\n",
    "axes[0].set_title('Flux Reconstruction (Masked positions in green)')\n",
    "axes[0].legend()\n",
    "axes[0].grid(True, alpha=0.3)\n",
    "\n",
    "# Error\n",
    "axes[1].plot(values[idx, :, 1].cpu().numpy(), 'b-', label='True', alpha=0.7)\n",
    "axes[1].plot(pred[idx, :, 1].cpu().numpy(), 'r--', label='Predicted', alpha=0.7)\n",
    "axes[1].scatter(np.where(mask_pos)[0], values[idx, mask_pos, 1].cpu().numpy(), \n",
    "               color='green', s=50, marker='o', label='Masked', zorder=5)\n",
    "axes[1].set_xlabel('Measurement Index')\n",
    "axes[1].set_ylabel('Error (arcsinh)')\n",
    "axes[1].set_title('Error Reconstruction (Masked positions in green)')\n",
    "axes[1].legend()\n",
    "axes[1].grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('sample_reconstruction.pdf', dpi=300, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Print summary table for paper\n",
    "print(\"\\\\begin{table}[h]\")\n",
    "print(\"\\\\centering\")\n",
    "print(\"\\\\begin{tabular}{|l|c|c|}\")\n",
    "print(\"\\\\hline\")\n",
    "print(\"Metric & Flux & Error \\\\\\\\ \\\\hline\")\n",
    "for metric in ['RMSE', 'MAE', 'R2', 'Explained Variance']:\n",
    "    print(f\"{metric} & {metrics['Flux'][metric]:.4f} & {metrics['Error'][metric]:.4f} \\\\\\\\\")\n",
    "print(\"\\\\hline\")\n",
    "print(\"\\\\end{tabular}\")\n",
    "print(f\"\\\\caption{{Reconstruction performance on validation set}}\")\n",
    "print(\"\\\\end{table}\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}